In [ ]:
import sys, pathlib
_root = next((c for c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (c / "projects" / "data").is_dir()), pathlib.Path.cwd())
sys.path.insert(0, str(_root))
from projects.data.make_synth_table import make_synth_table


# 06 · 可视化与探索性数据分析  Matplotlib & EDA

> **能力标签**：SH4（数据处理与分析）、SH5（数据工程）

## 目标 Objectives

完成本课后，你应该能够：

1. 正确设置 `matplotlib.use('Agg')` 无头后端，使图表在脚本 / CI / 无显示器环境中生成。
2. 用 `ax.plot` / `ax.scatter` / `ax.hist` 绘制折线图、散点图、直方图。
3. 用 `fig, axes = plt.subplots(m, n)` 创建子图网格，并添加标题、轴标签、图例。
4. 对 `make_synth_table()` 数据做简单 EDA（Exploratory Data Analysis）并保存图像。

> 对应能力：**SH4 · SH5**


## 概念讲解 Concepts

### 无头后端 Headless Backend

在服务器 / CI / Jupyter 无 GUI 环境运行时，必须在 **`import matplotlib.pyplot as plt` 之前** 设置后端：

```python
import matplotlib
matplotlib.use('Agg')   # 必须在 plt 之前！
import matplotlib.pyplot as plt
```

`'Agg'` 是纯软件光栅化后端，不依赖 X11 / GUI 库，将图表渲染到内存缓冲区，通过 `fig.savefig()` 保存。

---

### Figure / Axes 架构

| 对象 | 含义 |
|------|------|
| `Figure` | 整张画布（窗口）|
| `Axes` | 一个坐标系（子图）|

```python
fig, ax = plt.subplots()           # 单子图
fig, axes = plt.subplots(2, 3)     # 2 行 3 列
```

---

### 常用绘图方法

```python
ax.plot(x, y, label="曲线")         # 折线图
ax.scatter(x, y, c="blue", s=10)   # 散点图
ax.hist(data, bins=20)              # 直方图
ax.set_xlabel("X 轴")
ax.set_ylabel("Y 轴")
ax.set_title("图标题")
ax.legend()
fig.tight_layout()
fig.savefig("output.png", dpi=100)
plt.close(fig)                      # 释放内存（不用 plt.show()）
```

> **注意**：无头模式下**不要调用 `plt.show()`**，用 `fig.savefig()` 代替。


## 示例 Worked Example

In [ ]:
import matplotlib
matplotlib.use('Agg')   # 必须在 pyplot 之前！
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from projects.data.make_synth_table import make_synth_table

# 生成数据
df = make_synth_table(n=300, seed=42)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.suptitle('Synthetic Data EDA', fontsize=14)

# 1. 直方图：年龄分布
axes[0, 0].hist(df['age'], bins=20, color='steelblue', alpha=0.7)
axes[0, 0].set_title('年龄分布')
axes[0, 0].set_xlabel('age')
axes[0, 0].set_ylabel('频次')

# 2. 散点图：age vs x1
axes[0, 1].scatter(df['age'], df['x1'], alpha=0.3, s=10, color='teal')
axes[0, 1].set_title('age vs x1')
axes[0, 1].set_xlabel('age')
axes[0, 1].set_ylabel('x1')

# 3. 箱线图：x2 按 label 分组
for lbl, grp in df.groupby('label'):
    axes[1, 0].boxplot(grp['x2'].dropna(), positions=[lbl], widths=0.5)
axes[1, 0].set_title('x2 按 label')
axes[1, 0].set_xlabel('label')
axes[1, 0].set_ylabel('x2')

# 4. 条形图：group 分布
df['group'].value_counts().plot(kind='bar', ax=axes[1, 1], color='coral')
axes[1, 1].set_title('group 分布')
axes[1, 1].set_xlabel('group')
axes[1, 1].set_ylabel('频次')

import tempfile, os
_figdir = tempfile.mkdtemp()
plt.tight_layout()
plt.savefig(os.path.join(_figdir, 'eda.png'), dpi=80)
plt.close(fig)
print(f'图表已保存至 {_figdir}/eda.png')


## 动手 Exercise

在下面的 cell 中：

1. 生成 `make_synth_table(n=200, seed=5)` 数据。
2. 创建一个 `(1, 3)` 子图网格。
3. 子图 1：`x2` 直方图（bins=15）。
4. 子图 2：`age` vs `x3` 散点图。
5. 子图 3：`x1` 按 `label` 分组的箱线图。
6. 保存图像到临时目录。


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path
from projects.data.make_synth_table import make_synth_table

df = make_synth_table(n=200, seed=5)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Exercise EDA', fontsize=13)

# 1. x2 直方图
axes[0].hist(df['x2'], bins=15, color='coral', edgecolor='white')
axes[0].set_title('x2 分布')
axes[0].set_xlabel('x2')
axes[0].set_ylabel('频次')

# 2. age vs x3 散点图
axes[1].scatter(df['age'], df['x3'], alpha=0.4, s=10, color='steelblue')
axes[1].set_title('age vs x3')
axes[1].set_xlabel('age')
axes[1].set_ylabel('x3')

# 3. x1 按 label 箱线图
for lbl, grp in df.groupby('label'):
    axes[2].boxplot(grp['x1'].dropna(), positions=[lbl], widths=0.5)
axes[2].set_title('x1 按 label')
axes[2].set_xlabel('label')
axes[2].set_ylabel('x1')

import tempfile as _tmp, os
_figdir = _tmp.mkdtemp()
plt.tight_layout()
plt.savefig(os.path.join(_figdir, 'exercise_eda.png'), dpi=80)
plt.close(fig)
print(f'图表已保存至 {_figdir}/exercise_eda.png')


## 延伸阅读 Further Reading

1. **matplotlib 官方文档 — Backends**: <https://matplotlib.org/stable/users/explain/figure/backends.html>
2. **matplotlib 官方教程**: <https://matplotlib.org/stable/tutorials/index.html>
3. **《Python Data Science Handbook》第 4 章** — Jake VanderPlas（免费在线）
4. **Seaborn 统计可视化**: <https://seaborn.pydata.org/>


## 关联练习 Related Assignments

本课的可视化技能对应 **W3 数据管道小项目**（mini-project）`p-w3-data-pipeline`：

```bash
python check.py 05-data-pipeline
```

> 能力标签：**SH4 · SH5** · threshold ≥ 0.7
